# Exercises XP : Bases de donnees vectorielles et RAG

Ce notebook vous guide a travers la construction d'un pipeline RAG complet :
chargement des donnees -> embeddings -> FAISS -> ChromaDB -> Question-Answering.

**Ce que vous allez apprendre :**
- Generer des embeddings de phrases avec Sentence Transformers
- Indexer et rechercher des vecteurs avec FAISS
- Stocker et interroger des embeddings avec ChromaDB
- Construire un systeme de Q&A avec un modele Hugging Face (RAG)

**Important : executez les cellules dans l'ordre de haut en bas.**

## Exercice 0 - Installation des bibliotheques

Executez cette cellule **une seule fois** avant tout le reste.
Elle installe toutes les dependances necessaires pour les 5 exercices.

In [30]:
# ─────────────────────────────────────────────────────────────────
# INSTALLATION DES BIBLIOTHEQUES NECESSAIRES
# ─────────────────────────────────────────────────────────────────
# faiss-cpu       : bibliotheque de recherche vectorielle (Facebook AI)
# chromadb        : base de donnees vectorielle open-source
# sentence-transformers : generation d'embeddings de phrases
# transformers    : modeles de langage Hugging Face (GPT-2, BERT, etc.)
# datasets        : chargement de jeux de donnees Hugging Face

# libomp-dev est une dependance systeme necessaire pour FAISS sur Linux
!apt-get install -y -qq libomp-dev

# Installation des packages Python
%pip install -q faiss-cpu chromadb sentence-transformers transformers datasets pandas numpy

## Imports globaux

On importe ici toutes les bibliotheques utilisees dans ce notebook.
Cette cellule doit etre executee avant toute autre cellule de code.

In [31]:
# ─────────────────────────────────────────────────────────────────
# IMPORTS GLOBAUX
# ─────────────────────────────────────────────────────────────────
import os              # manipulation du systeme de fichiers
import json            # lecture/ecriture de donnees au format JSON
import numpy as np     # calcul numerique sur des tableaux
import pandas as pd    # manipulation de tableaux de donnees (DataFrames)
import faiss           # recherche vectorielle rapide

# SentenceTransformer : modele pour convertir du texte en vecteurs
# InputExample        : classe utilitaire pour formater les exemples
from sentence_transformers import SentenceTransformer, InputExample

# chromadb : base de donnees vectorielle
# Settings : configuration du client ChromaDB
import chromadb

# AutoTokenizer           : charge automatiquement le bon tokeniseur
# AutoModelForSeq2SeqLM   : modele seq2seq (encoder + decoder) pour la generation de texte
# pipeline                : interface simplifiee pour les modeles Hugging Face
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# display : affichage ameliore des DataFrames dans Jupyter
from IPython.display import display

# Creation d'un dossier 'cache' pour stocker les fichiers temporaires
# exist_ok=True : pas d'erreur si le dossier existe deja
os.makedirs('cache', exist_ok=True)

print('Tous les imports ont reussi.')

Tous les imports ont reussi.


## Exercice 1 - Chargement et preparation des donnees

Nous allons charger un jeu de donnees d'articles de presse etiquetes (`labelled_newscatcher_dataset.csv`).

**Etapes :**
1. Charger le CSV dans un DataFrame pandas
2. Ajouter une colonne identifiant unique (`id`) si elle n'existe pas
3. Inspecter les donnees (colonnes, types, valeurs manquantes)
4. Creer un sous-ensemble de 1000 lignes pour travailler plus rapidement

In [32]:
import os
import pandas as pd # Import pandas here as it's used directly in the code.
import datasets as hf_datasets # Import datasets here as it's used in the fallback.

# ─────────────────────────────────────────────────────────────────
# ETAPE 1 : Chargement du fichier CSV
# ─────────────────────────────────────────────────────────────────
# Remplacez le chemin ci-dessous par le vrai chemin de votre fichier CSV.
# Exemple sous Colab apres upload : '/content/labelled_newscatcher_dataset.csv'
# Exemple local : './data/labelled_newscatcher_dataset.csv'

# Option A : charger depuis un fichier CSV local (methode recommandee)
path = 'labelled_newscatcher_dataset.csv'  # <-- modifiez ce chemin si necessaire

# pd.read_csv() lit le fichier CSV et le charge dans un DataFrame
# Un DataFrame = tableau de donnees avec des lignes et des colonnes (comme Excel)
# sep=';' si les colonnes sont separees par des points-virgules (verifiez votre fichier)
try:
    # Check if the file actually exists before trying to read it
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found at {path}")
    pdf = pd.read_csv(path, sep=';')
    print(f'Fichier CSV charge avec succes. Dimensions : {pdf.shape}')
except FileNotFoundError:
    print(f"Fichier {path!r} introuvable. Veuillez telecharger le fichier 'labelled_newscatcher_dataset.csv' dans votre environnement Colab pour continuer. Pour l'instant, je ne peux pas charger le dataset depuis Hugging Face car toutes les tentatives ont echoue.")
    pdf = pd.DataFrame() # Create an empty DataFrame to avoid errors in subsequent steps if this block runs
    # The following Hugging Face fallback is commented out as it has consistently failed.
    # print(f'Fichier {path!r} introuvable. Tentative de chargement depuis Hugging Face...')
    # hf_ds = hf_datasets.load_dataset(
    #     'mlabonne/newscatcher-news-articles-2023', # Reverted to original dataset 'mlabonne/newscatcher-news-articles-2023'
    #     split='train'
    # )
    # pdf = hf_ds.to_pandas()
    # print(f'Dataset Hugging Face charge. Dimensions : {pdf.shape}')

# Only proceed with the rest of the steps if pdf is not empty (i.e., data was loaded)
if not pdf.empty:
    # ─────────────────────────────────────────────────────────────────
    # ETAPE 2 : Ajout d'une colonne identifiant unique
    # ─────────────────────────────────────────────────────────────────
    # Chaque ligne du DataFrame doit avoir un identifiant unique.
    # Cela facilite le suivi des enregistrements dans les bases de donnees vectorielles.
    # range(len(pdf)) genere une sequence 0, 1, 2, ..., N-1
    if 'id' not in pdf.columns:
        pdf['id'] = range(len(pdf))
        print('Colonne id creee.')
    else:
        print('Colonne id existante conservee.')

    # ─────────────────────────────────────────────────────────────────
    # ETAPE 3 : Inspection du jeu de donnees
    # ─────────────────────────────────────────────────────────────────
    print('\nApercu des 5 premieres lignes :')
    display(pdf.head())

    print('\nInformations sur les colonnes:')
    print(pdf.dtypes)

    print(f'\nNombre de valeurs manquantes par colonne :')
    print(pdf.isnull().sum())

    # ─────────────────────────────────────────────────────────────────
    # ETAPE 4 : Creation d'un sous-ensemble (subset) de 1000 lignes
    # ─────────────────────────────────────────────────────────────────
    # Travailler avec 1000 lignes au lieu de tout le dataset permet :
    # - d'accelerer les tests et le developpement
    # - de reduire la consommation memoire
    # Une fois le code valide, on peut l'appliquer au dataset complet.
    pdf_subset = pdf.head(1000).copy()
    print(f'\nSous-ensemble cree : {len(pdf_subset)} lignes')
    print('Colonnes disponibles :', pdf_subset.columns.tolist())
    display(pdf_subset[['id', 'title']].head())
else:
    print("Impossible de charger le dataset. Le reste du code depend de 'pdf' et 'pdf_subset'.")
    # Create an empty pdf_subset to prevent errors in subsequent cells if pdf is empty
    pdf_subset = pd.DataFrame()

Fichier 'labelled_newscatcher_dataset.csv' introuvable. Veuillez telecharger le fichier 'labelled_newscatcher_dataset.csv' dans votre environnement Colab pour continuer. Pour l'instant, je ne peux pas charger le dataset depuis Hugging Face car toutes les tentatives ont echoue.
Impossible de charger le dataset. Le reste du code depend de 'pdf' et 'pdf_subset'.


## Exercice 2 - Vectorisation avec Sentence Transformers

Les machines ne comprennent pas le texte brut. On doit le convertir en **vecteurs numeriques** (embeddings).

Un **embedding** est un tableau de nombres (ex: `[0.12, -0.45, 0.78, ...]`) qui encode
le sens semantique du texte. Deux phrases proches semantiquement auront des vecteurs proches
dans l'espace vectoriel.

Nous utilisons le modele `all-MiniLM-L6-v2` qui produit des vecteurs de **384 dimensions**.

In [33]:
# ─────────────────────────────────────────────────────────────────
# ETAPE 1 : Formatage des donnees avec InputExample
# ─────────────────────────────────────────────────────────────────
# InputExample est une classe utilitaire de sentence-transformers
# qui formate les entrees pour le modele d'embedding.
# Chaque exemple a :
#   - guid  : un identifiant unique (ici l'id de l'article)
#   - texts : la liste des textes a encoder (ici le titre de l'article)
#   - label : une etiquette numerique (ici 0.0 par defaut)

from sentence_transformers import InputExample

def example_create_fn(idx: int, text: str) -> InputExample:
    """
    Cree un objet InputExample a partir d'un identifiant et d'un texte.

    Parametres :
        idx  (int) : identifiant unique de l'article
        text (str) : titre de l'article a encoder

    Retourne :
        InputExample : objet formate pour le modele
    """
    return InputExample(
        guid=str(idx),   # identifiant unique (converti en chaine)
        texts=[text],    # liste de textes (ici un seul : le titre)
        label=0.0        # etiquette numerique (non utilisee pour l'inference)
    )

faiss_train_examples = [] # Initialize as empty list

if not pdf_subset.empty:
    # Application de la fonction sur tout le sous-ensemble
    # .apply() applique une fonction sur chaque ligne (axis=1) du DataFrame
    # lambda x: ... definit une fonction anonyme qui prend la ligne x en parametre
    # .tolist() convertit le resultat en liste Python
    faiss_train_examples = pdf_subset.apply(
        lambda x: example_create_fn(x['id'], x['title']),
        axis=1
    ).tolist()

    # Affichage des 2 premiers exemples pour verification
    print(f'Nombre d exemples crees : {len(faiss_train_examples)}')
    print('Exemple 0 :', faiss_train_examples[0])
    print('Exemple 1 :', faiss_train_examples[1])
else:
    print("Le DataFrame 'pdf_subset' est vide. Aucune exemple ne peut etre cree pour FAISS.")
    print(f'Nombre d exemples crees : {len(faiss_train_examples)}')

Le DataFrame 'pdf_subset' est vide. Aucune exemple ne peut etre cree pour FAISS.
Nombre d exemples crees : 0


In [34]:
# ─────────────────────────────────────────────────────────────────
# ETAPE 2 : Chargement du modele d'embedding
# ─────────────────────────────────────────────────────────────────
# SentenceTransformer charge un modele pre-entraine depuis Hugging Face.
# 'all-MiniLM-L6-v2' est un modele leger et performant :
#   - Taille : ~80 MB
#   - Dimensions des vecteurs : 384
#   - Bonne qualite pour la recherche semantique

model = SentenceTransformer('all-MiniLM-L6-v2')
print('Modele charge :', model)

# ─────────────────────────────────────────────────────────────────
# ETAPE 3 : Generation des embeddings
# ─────────────────────────────────────────────────────────────────
# On extrait les titres sous forme de liste de chaines de caracteres

if not pdf_subset.empty:
    titles_list = pdf_subset['title'].tolist()
    print(f'Nombre de titres a encoder : {len(titles_list)}')
    print('Exemple de titre :', titles_list[0])

    # model.encode() convertit chaque titre en un vecteur numerique
    # convert_to_numpy=True : retourne un tableau numpy (plus facile a utiliser avec FAISS)
    # show_progress_bar=True : affiche une barre de progression
    faiss_title_embedding = model.encode(
        titles_list,
        convert_to_numpy=True,
        show_progress_bar=True
    )

    # ─────────────────────────────────────────────────────────────────
    # ETAPE 4 : Verification des dimensions
    # ─────────────────────────────────────────────────────────────────
    # faiss_title_embedding est une matrice de forme (N, D) :
    #   N = nombre d'articles (1000)
    #   D = dimension des vecteurs (384 pour all-MiniLM-L6-v2)
    nb_embeddings = len(faiss_title_embedding)
    dim_embeddings = len(faiss_title_embedding[0])
    print(f'Nombre d embeddings generes : {nb_embeddings}')
    print(f'Dimension de chaque vecteur : {dim_embeddings}')
    print(f'Forme de la matrice complete : {faiss_title_embedding.shape}')
    print(f'Extrait du premier vecteur (10 premieres valeurs) : {faiss_title_embedding[0][:10]}')
else:
    print("Le DataFrame 'pdf_subset' est vide. Impossible de generer les embeddings.")
    # Initialize faiss_title_embedding to an empty numpy array to prevent errors in subsequent cells
    faiss_title_embedding = np.array([])
    # Also, we should ensure nb_embeddings and dim_embeddings are defined
    nb_embeddings = 0
    dim_embeddings = 0

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Modele charge : SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)
Le DataFrame 'pdf_subset' est vide. Impossible de generer les embeddings.


## Exercice 3 - Indexation et recherche avec FAISS

**FAISS** (Facebook AI Similarity Search) permet de trouver efficacement les vecteurs
les plus proches d'un vecteur de requete, meme dans des collections de millions d'elements.

**Principe :**
1. On normalise les vecteurs (longueur = 1)
2. On les ajoute a un index FAISS
3. Pour une requete, on encode le texte -> vecteur -> recherche les k plus proches voisins

**Pourquoi normaliser ?** La similarite cosinus (angle entre vecteurs) est equivalente
au produit scalaire (inner product) pour des vecteurs normalises. FAISS utilise
IndexFlatIP (inner product) pour cette raison.

In [35]:
import numpy as np
import faiss

# ─────────────────────────────────────────────────────────────────
# ETAPE 1 : Preparation des donnees pour l'indexation
# ─────────────────────────────────────────────────────────────────
# pdf_to_index : le DataFrame que l'on va indexer
# On garde le meme sous-ensemble que pour les embeddings
pdf_to_index = pdf_subset.copy()

# Check if pdf_to_index is empty
if not pdf_to_index.empty and len(faiss_title_embedding) > 0:
    # id_index : tableau des identifiants numeriques de chaque article
    # FAISS a besoin d'IDs entiers (int64) pour mapper les resultats aux articles
    # .to_numpy() convertit la colonne pandas en tableau numpy
    # .astype(np.int64) convertit les valeurs en entiers 64 bits
    id_index = pdf_to_index['id'].to_numpy().astype(np.int64)
    print(f'Nombre d articles a indexer : {len(id_index)}')
    print(f'Premiers IDs : {id_index[:5]}')

    # ─────────────────────────────────────────────────────────────────
    # ETAPE 2 : Normalisation des vecteurs
    # ─────────────────────────────────────────────────────────────────
    # faiss.normalize_L2() modifie le tableau EN PLACE (in-place)
    # Chaque vecteur est divise par sa norme pour avoir une longueur de 1
    # C'est obligatoire pour que le produit scalaire = similarite cosinus

    # On copie les embeddings en float32 (FAISS exige ce type)
    content_encoded_normalized = faiss_title_embedding.astype('float32')
    faiss.normalize_L2(content_encoded_normalized)  # normalisation en place
    print('Normalisation effectuee.')
    print(f'Norme du premier vecteur (doit etre ~1.0) : {np.linalg.norm(content_encoded_normalized[0]):.6f}')

    # ─────────────────────────────────────────────────────────────────
    # ETAPE 3 : Creation de l'index FAISS
    # ─────────────────────────────────────────────────────────────────
    # IndexFlatIP : index de type 'Flat Inner Product'
    #   - Flat = comparaison exhaustive (exacte mais plus lente que les index approximatifs)
    #   - IP   = Inner Product = produit scalaire (= similarite cosinus pour des vecteurs normalises)
    #   - Le parametre est la dimension des vecteurs

    # IndexIDMap : encapsule IndexFlatIP pour associer chaque vecteur a un ID personnalise
    #   - Sans IndexIDMap, FAISS utiliserait des indices 0, 1, 2...
    #   - Avec IndexIDMap, on peut associer nos vrais IDs d'articles

    dim = content_encoded_normalized.shape[1]  # dimension des vecteurs (384)
    index_content = faiss.IndexIDMap(faiss.IndexFlatIP(dim))

    # Ajout des vecteurs avec leurs IDs correspondants
    # add_with_ids(vecteurs, ids) : ajoute les vecteurs en les associant aux IDs
    index_content.add_with_ids(content_encoded_normalized, id_index)

    print(f'Index FAISS cree avec {index_content.ntotal} vecteurs.')
else:
    print("Le DataFrame 'pdf_to_index' est vide ou aucun embedding n'a ete genere. Impossible de creer l'index FAISS.")
    # Initialize index_content to None or an empty index to prevent errors in subsequent cells
    index_content = None
    # Initialize id_index for downstream cells to avoid potential errors
    id_index = np.array([])

Le DataFrame 'pdf_to_index' est vide ou aucun embedding n'a ete genere. Impossible de creer l'index FAISS.


In [36]:
# ─────────────────────────────────────────────────────────────────
# ETAPE 4 : Fonction de recherche dans l'index FAISS
# ─────────────────────────────────────────────────────────────────
def search_content(query: str, df: pd.DataFrame, k: int = 3) -> pd.DataFrame:
    """
    Recherche les k articles les plus similaires a une requete textuelle.

    Fonctionnement :
    1. Encode la requete en vecteur avec le meme modele que les articles
    2. Normalise le vecteur de requete
    3. Recherche les k plus proches voisins dans l'index FAISS
    4. Recupere les articles correspondants dans le DataFrame

    Parametres :
        query (str)       : la requete en langage naturel (ex: 'space exploration')
        df (DataFrame)    : le DataFrame contenant les articles a interroger
        k (int)           : nombre de resultats a retourner (defaut: 3)

    Retourne :
        DataFrame : les k articles les plus similaires avec leurs scores
    """
    global index_content # Declare index_content as global to ensure we are referring to the global variable

    if index_content is None or df.empty:
        print("L'index FAISS n'est pas initialise ou le DataFrame est vide. Impossible d'effectuer la recherche.")
        return pd.DataFrame(columns=df.columns.tolist() + ['similarities'])

    # ETAPE a : Encodage de la requete en vecteur numerique
    # model.encode() retourne une matrice de forme (1, 384)
    # .astype('float32') : conversion necessaire pour FAISS
    query_vector = model.encode([query]).astype('float32')

    # ETAPE b : Normalisation du vecteur de requete
    # Meme operation que pour les articles de l'index
    faiss.normalize_L2(query_vector)

    # ETAPE c : Recherche dans l'index FAISS
    # index_content.search(vecteur, k) retourne :
    #   - sims : scores de similarite (float, entre -1 et 1)
    #   - ids  : IDs des articles les plus similaires
    sims, ids = index_content.search(query_vector, k)

    # ETAPE d : Recuperation des articles correspondants
    # ids[0] : les IDs de la premiere (et seule) requete
    # df['id'].isin(ids[0]) : filtre les lignes dont l'ID est dans les resultats
    results = df[df['id'].isin(ids[0])].copy()

    # Ajout des scores de similarite dans une nouvelle colonne
    results['similarities'] = sims[0]

    # Tri par score decroissant (meilleur resultat en premier)
    results = results.sort_values('similarities', ascending=False)

    return results


# ─────────────────────────────────────────────────────────────────
# ETAPE 5 : Test de la recherche
# ─────────────────────────────────────────────────────────────────
# On teste avec la requete 'animal' et on attend les 5 articles les plus proches
print('Recherche des 5 articles les plus proches de la requete : animal')
print('=' * 60)
display(search_content('animal', pdf_to_index, k=5))

# On peut tester d'autres requetes
print('\nRecherche avec : space exploration')
print('=' * 60)
display(search_content('space exploration', pdf_to_index, k=3))

Recherche des 5 articles les plus proches de la requete : animal
L'index FAISS n'est pas initialise ou le DataFrame est vide. Impossible d'effectuer la recherche.


,similarities



Recherche avec : space exploration
L'index FAISS n'est pas initialise ou le DataFrame est vide. Impossible d'effectuer la recherche.


,similarities


## Exercice 4 - Collection ChromaDB et requetes

**ChromaDB** est une base de donnees vectorielle de plus haut niveau que FAISS.
Elle gere automatiquement la tokenisation, l'embedding et l'indexation.

**Differences cles avec FAISS :**
| Critere | FAISS | ChromaDB |
|-|-|-|
| Niveau | Bas niveau | Haut niveau |
| Embeddings | Manuels | Automatiques |
| Metadonnees | Non | Oui |
| Persistance | Fichier | Integree |
| Usage | Recherche pure | Application complete |

Dans ChromaDB, les donnees sont organisees en **collections** (semblables aux tables d'une BDD relationnelle).

In [37]:
# ─────────────────────────────────────────────────────────────────
# ETAPE 1 : Initialisation du client ChromaDB
# ─────────────────────────────────────────────────────────────────
import chromadb

# chromadb.Client() cree un client en memoire (ephemere)
# Les donnees ne sont pas sauvegardees sur disque entre les sessions
# Pour une persistance sur disque : chromadb.PersistentClient(path='./chroma_db')
chroma_client = chromadb.Client()
print('Client ChromaDB initialise.')

# ─────────────────────────────────────────────────────────────────
# ETAPE 2 : Creation de la collection
# ─────────────────────────────────────────────────────────────────
# Une collection = un espace de stockage pour un ensemble de documents
# Analogie : une table dans une base de donnees relationnelle

collection_name = 'my_news'

# On verifie si une collection du meme nom existe deja
# Si oui, on la supprime pour repartir de zero
# (evite les erreurs de doublons)
existing = [c.name for c in chroma_client.list_collections()]
if collection_name in existing:
    chroma_client.delete_collection(name=collection_name)
    print(f'Collection existante supprimee : {collection_name!r}')

# Creation d'une nouvelle collection vide
collection = chroma_client.create_collection(name=collection_name)
print(f'Collection creee : {collection_name!r}')

# ─────────────────────────────────────────────────────────────────
# ETAPE 3 : Ajout de documents dans la collection
# ─────────────────────────────────────────────────────────────────
# ChromaDB genere automatiquement les embeddings a partir des textes
# Il utilise en interne un modele d'embedding par defaut

# On ajoute les 100 premiers articles (pour aller plus vite)
# documents  : liste des textes a stocker (les titres des articles)
# metadatas  : informations supplementaires associees a chaque document
#              Ici on stocke le sujet (topic) de chaque article
# ids        : identifiants uniques OBLIGATOIRES pour chaque document
#              ChromaDB exige des chaines de caracteres (str)

nb_docs = 100  # nombre d'articles a ajouter

if not pdf_subset.empty:
    print(f'Ajout de {nb_docs} documents dans la collection...')
    print('(Cela peut prendre quelques secondes car ChromaDB genere les embeddings)')

    collection.add(
        documents=pdf_subset['title'][:nb_docs].tolist(),
        metadatas=[
            {'topic': str(topic)}
            for topic in pdf_subset['topic'][:nb_docs].tolist()
        ],
        ids=[str(x) for x in pdf_subset['id'][:nb_docs].tolist()]
    )
    print(f'Documents ajoutes. Collection contient : {collection.count()} documents.')
else:
    print("Le DataFrame 'pdf_subset' est vide. Impossible d'ajouter des documents a la collection ChromaDB.")

Client ChromaDB initialise.
Collection existante supprimee : 'my_news'
Collection creee : 'my_news'
Le DataFrame 'pdf_subset' est vide. Impossible d'ajouter des documents a la collection ChromaDB.


In [38]:
# ─────────────────────────────────────────────────────────────────
# ETAPE 4 : Interrogation de la collection ChromaDB
# ─────────────────────────────────────────────────────────────────
# collection.query() recherche les documents les plus similaires
# a une ou plusieurs requetes textuelles.

# ChromaDB s'occupe automatiquement de :
#   1. Convertir la requete en vecteur
#   2. Chercher les plus proches voisins
#   3. Retourner les documents et metadonnees correspondants

requete = 'space'   # mot-cle de recherche
n_resultats = 10    # nombre de documents a retourner

print(f'Requete ChromaDB : {requete!r} -> top {n_resultats} resultats')
print('=' * 60)

# query_texts : liste de requetes (on peut en envoyer plusieurs a la fois)
# n_results   : nombre de documents les plus proches a retourner
results = collection.query(
    query_texts=[requete],
    n_results=n_resultats
)

# Affichage du resultat formate en JSON
# results est un dictionnaire avec les cles : ids, documents, metadatas, distances
print(json.dumps(results, indent=4))

Requete ChromaDB : 'space' -> top 10 resultats
{
    "ids": [
        []
    ],
    "embeddings": null,
    "documents": [
        []
    ],
    "uris": null,
    "included": [
        "metadatas",
        "documents",
        "distances"
    ],
    "data": null,
    "metadatas": [
        []
    ],
    "distances": [
        []
    ]
}


## Exercice 5 - Question-Answering avec un modele Hugging Face

Nous combinons maintenant la **recherche documentaire** (ChromaDB) avec la
**generation de texte** (modele de langage) pour creer un systeme de Q&A.

C'est le principe du **RAG (Retrieval-Augmented Generation)** :
```
Question utilisateur
      |
      v
[ChromaDB] --> recupere des documents pertinents
      |
      v
[Prompt = contexte + question] --> envoye au modele de langage
      |
      v
Reponse generee par le modele
```

Nous utilisons **`google/flan-t5-small`**, un modele leger seq2seq optimise pour les taches
de question-answering et de raisonnement en contexte.

In [39]:
# ─────────────────────────────────────────────────────────────────
# ETAPE 1 : Chargement du modele de langage
# ─────────────────────────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# Choix du modele : google/flan-t5-small
# C'est un modele sequence-a-sequence (encoder + decoder)
# Avantages : leger (~300 MB), rapide, bon en Q&A sans fine-tuning
# Alternatives possibles : 'google/flan-t5-base' (plus grand, meilleur)
model_id = 'google/flan-t5-small'

print(f'Chargement du tokeniseur pour : {model_id}')
# AutoTokenizer charge automatiquement le tokeniseur adapte au modele
# Le tokeniseur convertit le texte brut en tokens numeriques
tokenizer = AutoTokenizer.from_pretrained(model_id)

print(f'Chargement du modele : {model_id}')
# AutoModelForSeq2SeqLM charge un modele de type encoder-decoder
# (adapte aux taches de traduction, resume, Q&A)
# device_map='auto' utilise le GPU si disponible, sinon le CPU
lm_model = AutoModelForSeq2SeqLM.from_pretrained(model_id, device_map='auto')
print('Modele charge avec succes.')

Chargement du tokeniseur pour : google/flan-t5-small
Chargement du modele : google/flan-t5-small


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Modele charge avec succes.


In [40]:
# ─────────────────────────────────────────────────────────────────
# ETAPE 2 : Creation d'une fonction de generation de reponses
# ─────────────────────────────────────────────────────────────────
# Comme la tache 'text2text-generation' n'est pas reconnue par le pipeline
# dans cet environnement, nous allons utiliser directement le modele
# et le tokeniseur pour generer la reponse.

def generate_flan_response(prompt: str, model, tokenizer, max_new_tokens: int = 256) -> str:
    """
    Genere une reponse a partir d'un prompt en utilisant le modele Flan-T5.
    """
    # Tokenisation du prompt
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    # Generation de la reponse
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)

    # Decodage de la reponse
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

print('Fonction de generation de reponse cree.')

# La variable `pipe` n'est plus necessaire, mais nous laissons la ligne commentee
# pour reference si le pipeline devait etre reintroduit dans un autre contexte.
# pipe = pipeline(
#     'text2text-generation',
#     model=lm_model,
#     tokenizer=tokenizer,
#     max_new_tokens=256,
#     device_map='auto',
# )

Fonction de generation de reponse cree.


In [41]:
# ─────────────────────────────────────────────────────────────────
# ETAPE 3 : Requete ChromaDB pour recuperer le contexte
# ─────────────────────────────────────────────────────────────────
# Avant de poser une question au modele, on recupere les documents
# les plus pertinents depuis ChromaDB pour alimenter le contexte.

# La question de l'utilisateur
question = "What's the latest news on space development?"

# Recherche des 5 articles les plus pertinents dans ChromaDB
# (on utilise les resultats de la requete de l'exercice precedent
#  ou on en fait une nouvelle avec la question)
chroma_results = collection.query(
    query_texts=[question],
    n_results=5
)

# Recuperation des documents trouves
# chroma_results['documents'] est une liste de listes (une par requete)
# chroma_results['documents'][0] = documents pour la premiere requete
context_docs = chroma_results['documents'][0]

print(f'Question : {question}')
print(f'Nombre de documents recuperes : {len(context_docs)}')
print('\nDocuments utilises comme contexte :')
for i, doc in enumerate(context_docs, 1):
    print(f'  {i}. {doc}')

Question : What's the latest news on space development?
Nombre de documents recuperes : 0

Documents utilises comme contexte :


In [42]:
# ─────────────────────────────────────────────────────────────────
# ETAPE 4 : Construction du prompt et generation de la reponse
# ─────────────────────────────────────────────────────────────────
# On construit un prompt qui combine le contexte et la question.
# Le modele va utiliser le contexte pour generer une reponse pertinente.

# Concatenation des documents en un seul texte de contexte
# On numerate chaque document (#1, #2, etc.) pour la lisibilite
context = ' '.join([f'#{i} {doc}' for i, doc in enumerate(context_docs, 1)])

# Template du prompt : contexte + question
# Le modele flan-t5 est entraine a repondre a des questions
# basees sur un contexte donne (reading comprehension)
prompt = (
    f'Relevant context: {context}\n\n'
    f"The user's question: {question}\n"
    f'Answer:'
)

print('Prompt envoye au modele :')
print('-' * 60)
print(prompt[:500], '...' if len(prompt) > 500 else '')
print('-' * 60)

# Generation de la reponse en utilisant la fonction generate_flan_response
print('\nGeneration de la reponse...')
lm_response = generate_flan_response(prompt, lm_model, tokenizer, max_new_tokens=256)

print('\nReponse generee par le modele :')
print('=' * 60)
print(lm_response)
print('=' * 60)

Prompt envoye au modele :
------------------------------------------------------------
Relevant context: 

The user's question: What's the latest news on space development?
Answer: 
------------------------------------------------------------

Generation de la reponse...

Reponse generee par le modele :
Science/Tech


In [43]:
# ─────────────────────────────────────────────────────────────────
# ETAPE 5 : Experimentation avec differentes questions
# ─────────────────────────────────────────────────────────────────
# On peut tester d'autres questions et varier le nombre de documents
# de contexte pour observer l'impact sur les reponses.

def ask_question(question: str, n_context_docs: int = 3) -> str:
    """
    Pose une question au systeme RAG et retourne la reponse.

    Parametres :
        question (str)        : la question en langue naturelle
        n_context_docs (int)  : nombre de documents de contexte a utiliser

    Retourne :
        str : la reponse generee par le modele
    """
    # 1. Recherche des documents pertinents dans ChromaDB
    chroma_res = collection.query(
        query_texts=[question],
        n_results=n_context_docs
    )
    docs = chroma_res['documents'][0]

    # 2. Construction du contexte et du prompt
    context = ' '.join([f'#{i} {d}' for i, d in enumerate(docs, 1)])
    prompt = (
        f'Relevant context: {context}\n\n'
        f"The user's question: {question}\n"
        f'Answer:'
    )

    # 3. Generation de la reponse
    response = generate_flan_response(prompt, lm_model, tokenizer, max_new_tokens=256)
    return response


# Tests avec differentes questions et tailles de contexte
questions_test = [
    ('What is happening in the economy?', 3),
    ('Tell me about sports news', 5),
    ('Any news about technology companies?', 2),
]

for q, n_ctx in questions_test:
    print(f'Question ({n_ctx} docs contexte) : {q}')
    reponse = ask_question(q, n_context_docs=n_ctx)
    print(f'Reponse : {reponse}')
    print('-' * 60)

# ANALYSE : Que remarque-t-on ?
print("""
ANALYSE DES RESULTATS RAG :

1. Avec plus de documents de contexte (n_context_docs plus grand) :
   -> La reponse peut etre plus riche en informations
   -> Mais le prompt devient plus long (risque de depasser la limite du modele)

2. Avec un modele plus petit (flan-t5-small) :
   -> Les reponses sont courtes et parfois generiques
   -> Un modele plus grand (flan-t5-base ou flan-t5-large) donnera de meilleures reponses

3. Limitation cle :
   -> Notre collection ne contient que 100 articles
   -> Avec plus d articles, les resultats de recherche seraient plus pertinents
""")

Question (3 docs contexte) : What is happening in the economy?
Reponse : economy
------------------------------------------------------------
Question (5 docs contexte) : Tell me about sports news
Reponse : Sports
------------------------------------------------------------
Question (2 docs contexte) : Any news about technology companies?
Reponse : Science/Tech
------------------------------------------------------------

ANALYSE DES RESULTATS RAG :

1. Avec plus de documents de contexte (n_context_docs plus grand) :
   -> La reponse peut etre plus riche en informations
   -> Mais le prompt devient plus long (risque de depasser la limite du modele)

2. Avec un modele plus petit (flan-t5-small) :
   -> Les reponses sont courtes et parfois generiques
   -> Un modele plus grand (flan-t5-base ou flan-t5-large) donnera de meilleures reponses

3. Limitation cle :
   -> Notre collection ne contient que 100 articles
   -> Avec plus d articles, les resultats de recherche seraient plus pertinent

## Recapitulatif

Vous avez construit un pipeline RAG complet :

```
Donnees CSV
    -> pandas DataFrame (Exercice 1)
    -> Embeddings numeriques via SentenceTransformer (Exercice 2)
    -> Index FAISS pour la recherche vectorielle rapide (Exercice 3)
    -> Collection ChromaDB pour le stockage et la recherche documentaire (Exercice 4)
    -> Question-Answering avec flan-t5 + contexte ChromaDB (Exercice 5)
```

**Concepts cles a retenir :**
- Un **embedding** = representation numerique du sens d'un texte
- **FAISS** = recherche vectorielle bas niveau, rapide, en memoire
- **ChromaDB** = base de donnees vectorielle haut niveau, avec metadonnees
- **RAG** = Retrieval-Augmented Generation = recuperer des documents + generer une reponse
- **Flan-T5** = modele seq2seq pre-entraine pour le raisonnement et le Q&A